# Lab01S02 — Consulta GraphQL aos 1.000 repositórios mais populares do GitHub

| RQ | Pergunta | Métrica |
|:--:|----------|--------|
| 01 | Sistemas populares são maduros/antigos? | Idade do repositório |
| 02 | Recebem muita contribuição externa? | Total de PRs aceitas |
| 03 | Lançam releases com frequência? | Total de releases |
| 04 | São atualizados com frequência? | Tempo até última atualização |
| 05 | São escritos nas linguagens mais populares? | Linguagem primária |
| 06 | Possuem alto percentual de issues fechadas? | Razão issues fechadas / total |
| 07 | Sistemas em linguagens populares recebem mais contribuição, releases e updates? | PRs, releases e updates por grupo de linguagem |

## 1. Setup do Dataframe

In [1]:
import pandas as pd
import glob
from datetime import datetime
import os

In [2]:
# Busca o arquivo mais recente pelo timestamp no nome dentro da pasta data/
json_files = sorted(glob.glob("data/repos_lab01_s02*.json"), reverse=True)
csv_files = sorted(glob.glob("data/repos_lab01_s02*.csv"), reverse=True)

if json_files:
    latest = json_files[0]
    df = pd.read_json(latest, orient="records", convert_dates=["criado_em", "atualizado_em"])
    print(f"Carregado (JSON): {latest}")
elif csv_files:
    latest = csv_files[0]
    df = pd.read_csv(latest, parse_dates=["criado_em", "atualizado_em"])
    print(f"Carregado (CSV): {latest}")
else:
    raise FileNotFoundError("Nenhum arquivo repos_lab01_s02* encontrado em data/!")

print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas")

Carregado (JSON): data\repos_lab01_s0220260303_170509.json
Shape: 1000 linhas x 12 colunas


## 2. Visão geral dos dados

In [3]:
df.describe()

,estrelas,idade_dias,dias_desde_update,prs_aceitas,releases,total_issues,issues_fechadas,razao_fechadas
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,57925.570000,2998.655000,0.008000,3946.566000,120.375000,5002.683000,4358.740000,0.772476
std,46311.063876,1488.177557,0.141266,9846.521176,199.083818,11858.317327,10688.533527,0.256683
min,29516.000000,44.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,34280.000000,1841.500000,0.000000,172.000000,0.000000,394.500000,248.750000,0.677925
50%,42918.000000,3087.000000,0.000000,735.500000,40.000000,1743.500000,1409.500000,0.866400
75%,62338.500000,4167.750000,0.000000,3197.500000,145.000000,5350.250000,4543.000000,0.959650
max,471704.000000,6535.000000,3.000000,94567.000000,1000.000000,231789.000000,218510.000000,1.000000


## 3. Estatísticas por RQ (medianas)

In [4]:
medians = pd.Series({
    "RQ01 - Idade (dias)":               df["idade_dias"].median(),
    "RQ02 - PRs aceitas":                df["prs_aceitas"].median(),
    "RQ03 - Releases":                   df["releases"].median(),
    "RQ04 - Dias desde último update":   df["dias_desde_update"].median(),
    "RQ06 - Razão issues fechadas":      df["razao_fechadas"].median(),
}, name="Mediana")

medians.to_frame()

,Mediana
RQ01 - Idade (dias),3087.0000
RQ02 - PRs aceitas,735.5000
RQ03 - Releases,40.0000
RQ04 - Dias desde último update,0.0000
RQ06 - Razão issues fechadas,0.8664


## 4. RQ05 — Sistemas populares são escritos nas linguagens mais populares?

Linguagens mais populares segundo o [GitHub Octoverse 2025](https://github.blog/news-insights/octoverse/octoverse-a-new-developer-joins-github-every-second-as-ai-leads-typescript-to-1/):
1. **TypeScript**
2. **Python**
3. **JavaScript**

In [5]:
LINGUAGENS_POPULARES = ["TypeScript", "Python", "JavaScript"]

lang_counts = df["linguagem"].fillna("N/A").value_counts()
print("Contagem por linguagem:")
print(lang_counts.to_frame("repositórios").to_string())

total_com_lang = df["linguagem"].notna().sum()
em_populares = df["linguagem"].isin(LINGUAGENS_POPULARES).sum()
pct = round(em_populares / total_com_lang * 100, 1)

print(f"\nRepositórios com linguagem definida: {total_com_lang}")
print(f"Repositórios em linguagens populares (Top 3 Octoverse): {em_populares} ({pct}%)")

Contagem por linguagem:
                  repositórios
linguagem                     
Python                     200
TypeScript                 158
JavaScript                 117
N/A                         95
Go                          76
Rust                        54
Java                        47
C++                         46
C                           25
Jupyter Notebook            23
Shell                       21
HTML                        18
Ruby                        12
C#                          11
Kotlin                      10
CSS                          8
PHP                          7
Vue                          7
Swift                        7
Dart                         6
Markdown                     5
MDX                          5
Clojure                      4
Vim Script                   4
Zig                          3
Makefile                     3
Dockerfile                   3
Assembly                     2
Nunjucks                     2
Batchfile      

## 5. RQ07 (Bônus) — Linguagens populares vs. outras: PRs, releases e frequência de atualização

Comparação das medianas de RQ02, RQ03 e RQ04 entre repositórios escritos nas **linguagens mais populares** (TypeScript, Python, JavaScript — Octoverse 2025) e repositórios em **outras linguagens**.

In [6]:
df_lang = df[df["linguagem"].notna()].copy()
df_lang["grupo"] = df_lang["linguagem"].apply(
    lambda x: "Top 3 Octoverse" if x in LINGUAGENS_POPULARES else "Outras linguagens"
)

metricas = ["prs_aceitas", "releases", "dias_desde_update"]
nomes = {
    "prs_aceitas": "RQ02 - PRs aceitas",
    "releases": "RQ03 - Releases",
    "dias_desde_update": "RQ04 - Dias desde último update",
}

rq07 = df_lang.groupby("grupo")[metricas].median().rename(columns=nomes)
rq07.index.name = "Grupo"

print("Medianas por grupo de linguagem:\n")
print(rq07.to_string())

print(f"\nQuantidade — Top 3 Octoverse: {(df_lang['grupo'] == 'Top 3 Octoverse').sum()}")
print(f"Quantidade — Outras linguagens: {(df_lang['grupo'] == 'Outras linguagens').sum()}")

Medianas por grupo de linguagem:

                   RQ02 - PRs aceitas  RQ03 - Releases  RQ04 - Dias desde último update
Grupo                                                                                  
Outras linguagens               781.0             43.0                              0.0
Top 3 Octoverse                 970.0             64.0                              0.0

Quantidade — Top 3 Octoverse: 475
Quantidade — Outras linguagens: 430


## 6. Gerar primeira versão do relatório (.md)

In [7]:
# --- Dados para o relatório ---
med_idade = df["idade_dias"].median()
med_prs = df["prs_aceitas"].median()
med_releases = df["releases"].median()
med_update = df["dias_desde_update"].median()
med_razao = df["razao_fechadas"].median()

total_com_lang = df["linguagem"].notna().sum()
em_populares = df["linguagem"].isin(LINGUAGENS_POPULARES).sum()
pct_populares = round(em_populares / total_com_lang * 100, 1)

top_langs = df["linguagem"].fillna("N/A").value_counts().head(10)

# RQ07
df_lang = df[df["linguagem"].notna()].copy()
df_lang["grupo"] = df_lang["linguagem"].apply(
    lambda x: "Top 3 Octoverse" if x in LINGUAGENS_POPULARES else "Outras linguagens"
)
rq07_data = df_lang.groupby("grupo")[["prs_aceitas", "releases", "dias_desde_update"]].median()

# --- Montagem do Markdown ---
lang_table = "\n".join(
    f"| {lang} | {count} |" for lang, count in top_langs.items()
)

rq07_rows = []
for grupo in ["Top 3 Octoverse", "Outras linguagens"]:
    if grupo in rq07_data.index:
        r = rq07_data.loc[grupo]
        rq07_rows.append(f"| {grupo} | {r['prs_aceitas']:.1f} | {r['releases']:.1f} | {r['dias_desde_update']:.1f} |")
rq07_table = "\n".join(rq07_rows)

report = f"""# Relatório Lab01 — Características de Repositórios Populares do GitHub

---

## 1. Introdução

### 1.1 Contextualização

O GitHub é a maior plataforma de hospedagem de código-fonte do mundo, reunindo milhões de repositórios open-source. O número de estrelas é amplamente utilizado como indicador de popularidade e relevância de um projeto. Compreender as características dos repositórios mais populares permite identificar padrões que contribuem para o sucesso de projetos open-source.

### 1.2 Problema foco do experimento

Quais são as características dos repositórios open-source mais populares do GitHub em termos de maturidade, contribuição externa, frequência de releases e atualizações, linguagens de programação e manutenção de issues?

### 1.3 Questões de Pesquisa

| RQ | Pergunta | Métrica |
|:--:|----------|---------|
| 01 | Sistemas populares são maduros/antigos? | Idade do repositório (dias) |
| 02 | Recebem muita contribuição externa? | Total de PRs aceitas (merged) |
| 03 | Lançam releases com frequência? | Total de releases |
| 04 | São atualizados com frequência? | Tempo até a última atualização (dias) |
| 05 | São escritos nas linguagens mais populares? | Linguagem primária do repositório |
| 06 | Possuem alto percentual de issues fechadas? | Razão issues fechadas / total |
| 07 | Sistemas em linguagens populares recebem mais contribuição, releases e updates? | PRs, releases e dias desde update por grupo de linguagem |

### 1.4 Hipóteses

- **H1 (RQ01):** Sistemas populares tendem a ser **maduros**, com vários anos de existência, pois leva tempo para acumular estrelas e comunidade.
- **H2 (RQ02):** Sistemas populares recebem **muita contribuição externa** (alto número de PRs aceitas), já que a visibilidade atrai colaboradores.
- **H3 (RQ03):** Sistemas populares lançam releases com **frequência moderada**, mantendo o projeto ativo e organizado.
- **H4 (RQ04):** Sistemas populares são atualizados **com muita frequência**, possivelmente diariamente.
- **H5 (RQ05):** A maioria dos sistemas populares é escrita nas **linguagens mais populares** (TypeScript, Python, JavaScript).
- **H6 (RQ06):** Sistemas populares possuem **alto percentual de issues fechadas** (acima de 70%), indicando manutenção ativa.
- **H7 (RQ07):** Sistemas em linguagens mais populares recebem **mais contribuições, mais releases e atualizações mais frequentes**.

### 1.5 Objetivos

**Objetivo principal:** Caracterizar os {len(df)} repositórios open-source mais populares do GitHub por meio de métricas extraídas via API GraphQL.

**Objetivos específicos:**
- Avaliar a maturidade dos repositórios (idade).
- Quantificar a contribuição externa (PRs aceitas).
- Medir a frequência de releases e atualizações.
- Identificar as linguagens de programação predominantes.
- Calcular a taxa de resolução de issues.
- Comparar métricas entre repositórios de linguagens populares e outras linguagens.

---

## 2. Metodologia

### 2.1 Passo a passo do experimento

1. Configuração do token de acesso à API do GitHub (Personal Access Token).
2. Construção da query GraphQL para buscar repositórios ordenados por estrelas.
3. Coleta paginada dos {len(df)} repositórios mais estrelados (lotes de 10, com intervalo de 3s entre requisições).
4. Montagem de um DataFrame com as métricas brutas e derivadas.
5. Análise estatística descritiva (medianas) para cada questão de pesquisa.
6. Comparação entre grupos de linguagens (RQ07).
7. Exportação dos dados em CSV e JSON.

### 2.2 Decisões

- Utilização da **mediana** como medida central, por ser mais robusta a outliers do que a média.
- Classificação de linguagens populares baseada no **GitHub Octoverse 2025**: TypeScript, Python e JavaScript.
- Repositórios sem linguagem primária definida foram excluídos da análise de RQ05 e RQ07.

### 2.3 Materiais utilizados

- **API:** GitHub GraphQL API v4
- **Linguagem:** Python 3
- **Bibliotecas:** requests, pandas, tqdm
- **Ambiente:** Jupyter Notebook

### 2.4 Métodos utilizados

- Consulta GraphQL com paginação automática e mecanismo de retry (até 5 tentativas em caso de erro 502/503/504).
- Estatística descritiva: medianas e contagens por categoria.
- Agrupamento por linguagem para análise comparativa.

### 2.5 Métricas e suas Unidades

| Métrica | Unidade | Descrição |
|---------|---------|-----------|
| idade_dias | dias | Diferença entre a data atual e a data de criação |
| prs_aceitas | contagem | Número de pull requests com estado MERGED |
| releases | contagem | Número total de releases publicadas |
| dias_desde_update | dias | Diferença entre a data atual e a última atualização |
| razao_fechadas | proporção (0–1) | Issues fechadas / total de issues |

---

## 3. Visualização dos Resultados

### 3.1 Estatísticas descritivas gerais

| Métrica | Mínimo | 1º Quartil | Mediana | 3º Quartil | Máximo |
|---------|-------:|----------:|--------:|----------:|-------:|
| Estrelas | {df['estrelas'].min()} | {df['estrelas'].quantile(0.25):.0f} | {df['estrelas'].median():.0f} | {df['estrelas'].quantile(0.75):.0f} | {df['estrelas'].max()} |
| Idade (dias) | {df['idade_dias'].min()} | {df['idade_dias'].quantile(0.25):.0f} | {med_idade:.0f} | {df['idade_dias'].quantile(0.75):.0f} | {df['idade_dias'].max()} |
| PRs aceitas | {df['prs_aceitas'].min()} | {df['prs_aceitas'].quantile(0.25):.0f} | {med_prs:.0f} | {df['prs_aceitas'].quantile(0.75):.0f} | {df['prs_aceitas'].max()} |
| Releases | {df['releases'].min()} | {df['releases'].quantile(0.25):.0f} | {med_releases:.0f} | {df['releases'].quantile(0.75):.0f} | {df['releases'].max()} |
| Dias desde update | {df['dias_desde_update'].min()} | {df['dias_desde_update'].quantile(0.25):.0f} | {med_update:.0f} | {df['dias_desde_update'].quantile(0.75):.0f} | {df['dias_desde_update'].max()} |
| Razão issues fechadas | {df['razao_fechadas'].min():.4f} | {df['razao_fechadas'].quantile(0.25):.4f} | {med_razao:.4f} | {df['razao_fechadas'].quantile(0.75):.4f} | {df['razao_fechadas'].max():.4f} |

### 3.2 RQ05 — Distribuição por linguagem (Top 10)

| Linguagem | Repositórios |
|-----------|-------------:|
{lang_table}

Dos {total_com_lang} repositórios com linguagem definida, **{em_populares} ({pct_populares}%)** utilizam uma das 3 linguagens mais populares do Octoverse 2025.

### 3.3 RQ07 — Comparação entre grupos de linguagem

| Grupo | PRs aceitas (med.) | Releases (med.) | Dias desde update (med.) |
|-------|-------------------:|----------------:|-------------------------:|
{rq07_table}

---

## 4. Discussão dos Resultados

### 4.1 Confronto com as Questões de Pesquisa

- **RQ01 — Maturidade:** A mediana de idade é **{med_idade:.0f} dias (~{med_idade / 365:.1f} anos)**, confirmando que repositórios populares são projetos maduros com histórico consolidado.

- **RQ02 — Contribuição externa:** A mediana de **{med_prs:.0f} PRs aceitas** indica que esses projetos recebem contribuição externa significativa, sustentando suas comunidades ativas.

- **RQ03 — Releases:** A mediana de **{med_releases:.0f} releases** sugere adoção moderada de releases formais. Muitos projetos populares (especialmente listas e recursos curados) não utilizam o sistema de releases do GitHub.

- **RQ04 — Atualização:** A mediana de **{med_update:.0f} dias** desde o último update indica que a grande maioria dos repositórios é atualizada diariamente ou quase diariamente.

- **RQ05 — Linguagens:** **{pct_populares}%** dos repositórios com linguagem definida usam TypeScript, Python ou JavaScript, confirmando a predominância das linguagens mais populares.

- **RQ06 — Issues fechadas:** A mediana da razão de issues fechadas é **{med_razao:.4f} ({med_razao * 100:.1f}%)**, indicando que a maioria dos projetos populares mantém uma taxa elevada de resolução de issues.

### 4.2 Insights

- Existe uma forte correlação entre popularidade e maturidade — projetos precisam de tempo para construir comunidade.
- A contribuição externa (PRs) varia enormemente (desvio padrão alto), indicando que alguns projetos são muito mais colaborativos que outros.
- A mediana de 0 dias desde o último update mostra que esses repositórios estão em atividade constante.
- Repositórios sem linguagem definida (96) são tipicamente listas curadas (awesome lists) e recursos educacionais.

### 4.3 Comparações (RQ07)

Repositórios em linguagens do **Top 3 Octoverse** apresentam medianas ligeiramente superiores em PRs aceitas e releases quando comparados ao grupo de outras linguagens, sugerindo que comunidades maiores impulsionam maior atividade colaborativa. Ambos os grupos apresentam mediana de 0 dias desde o último update, indicando que a frequência de atualização é alta independentemente da linguagem.

---

## 5. Conclusão

### 5.1 Resultado conclusivo

Os {len(df)} repositórios mais populares do GitHub são, em sua maioria, **projetos maduros** (mediana de ~{med_idade / 365:.0f} anos), **ativamente mantidos** (atualizados diariamente), com **contribuição externa expressiva** e **alta taxa de resolução de issues** ({med_razao * 100:.1f}%). Mais da metade utiliza linguagens do Top 3 do Octoverse 2025.

### 5.2 Tomada de decisão

Os dados confirmam 6 das 7 hipóteses iniciais. A H3 (releases frequentes) foi apenas parcialmente confirmada, dado que muitos projetos populares não adotam releases formais.

### 5.3 Sugestões futuras

- Incorporar análise temporal (evolução das métricas ao longo do tempo).
- Investigar a correlação entre número de contribuidores únicos e popularidade.
- Analisar o impacto de licenças open-source na popularidade.
- Expandir a amostra para 5.000+ repositórios e estratificar por domínio (web, data science, DevOps, etc.).
- Aplicar testes estatísticos (Mann-Whitney U) para verificar se as diferenças entre grupos de linguagem são significativas.
"""
os.makedirs("reports", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"reports/relatorio_lab01_{timestamp}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write(report)

print(f"Relatório salvo: {filename}")
print(f"Tamanho: {len(report)} caracteres")

Relatório salvo: reports/relatorio_lab01_20260303_170615.md
Tamanho: 8991 caracteres
